# Turkish GPT-2 LoRA fine-tuning

Install `requirements.txt` in an isolated environment and select its Jupyter kernel before running this notebook. Start Jupyter from the repository root or `code/`. Each run trains one selected model and dataset. Set `RUN_TRAINING = True` only when you intend to download base weights and train. The checked-in plots and output samples are historical artifacts; these cells have no saved execution results.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "Data").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "Data").is_dir() or not (ROOT / "code" / "qa_training.py").is_file():
    raise RuntimeError("Start Jupyter from the repository root or code directory")
sys.path.insert(0, str(ROOT / "code"))

from qa_training import load_qa_data, tokenize_qa_batch, generate_answer

MODEL_SIZE = "medium"
DATASET_VERSION = "v1"
MODEL_REVISIONS = {
    "medium": "169f7b8cd4e369b96eb2739b46345c4ecb2b9d4a",
    "large": "c4ddd0f3808fb7bf23ab63d9fea6ec98c1899934",
}
MODEL_REVISION = MODEL_REVISIONS.get(MODEL_SIZE)
RUN_TRAINING = False
SEED = 42
MAX_LENGTH = 512
MODELS = {
    "large": "ytu-ce-cosmos/turkish-gpt2-large",
    "medium": "ytu-ce-cosmos/turkish-gpt2-medium",
}
if MODEL_SIZE not in MODELS:
    raise ValueError("MODEL_SIZE must be medium or large")
if DATASET_VERSION not in {"v1", "v2", "v3"}:
    raise ValueError("DATASET_VERSION must be v1, v2, or v3")
if not isinstance(MODEL_REVISION, str) or len(MODEL_REVISION) != 40 or not set(MODEL_REVISION.lower()) <= set("0123456789abcdef"):
    raise ValueError("MODEL_REVISION must be a full 40-character Hub commit")
RUN_NAME = f"{MODEL_SIZE}_{DATASET_VERSION}"


In [ ]:
train_examples = load_qa_data(ROOT / "Data", DATASET_VERSION)
test_examples = load_qa_data(ROOT / "Data", "test")
print(f"Training pairs: {len(train_examples)}; qualitative test pairs: {len(test_examples)}")


## Load one base model

The next cell downloads model weights when they are not cached. Medium and large training require substantial RAM or GPU memory. CPU execution disables FP16 but is still expensive. Set `MODEL_REVISION` to a Hub commit for repeatable base weights. The resolved commit and installed package versions are recorded with the saved adapter.


In [ ]:
if not RUN_TRAINING:
    raise RuntimeError("Set RUN_TRAINING = True to enable this local training run")

import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, default_data_collator, set_seed
from peft import LoraConfig, get_peft_model

set_seed(SEED)
model_path = MODELS[MODEL_SIZE]
model = AutoModelForCausalLM.from_pretrained(model_path, revision=MODEL_REVISION, use_safetensors=True)
resolved_revision = MODEL_REVISION
tokenizer = AutoTokenizer.from_pretrained(model_path, revision=resolved_revision)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
if not 2 <= MAX_LENGTH <= model.config.max_position_embeddings:
    raise ValueError("MAX_LENGTH must fit the selected model context")
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.1,
    bias="none",
    revision=resolved_revision,
)
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()


In [ ]:
train_dataset = Dataset.from_list(train_examples)
train_dataset = train_dataset.map(
    tokenize_qa_batch,
    batched=True,
    fn_kwargs={"tokenizer": tokenizer, "max_length": MAX_LENGTH},
    remove_columns=train_dataset.column_names,
)


## Train and save

Each causal sequence contains `Soru: ... Cevap: ...` followed by EOS. Labels match that sequence; only padding has label `-100`. Training loss covers the whole sequence. `Data/test.csv` is used only for qualitative generation, not validation loss or an accuracy metric. Long sequences are truncated to `MAX_LENGTH - 1` text tokens plus EOS and can lose part of an answer.


In [ ]:
training_args = TrainingArguments(
    output_dir=str(ROOT / "results" / RUN_NAME),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    logging_steps=50,
    save_steps=10000,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    data_collator=default_data_collator,
)
trainer.train()


In [ ]:
import hashlib
import importlib.metadata
import json

save_path = ROOT / "saved_models" / RUN_NAME
peft_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
run_metadata = {
    "model": model_path,
    "model_revision": resolved_revision,
    "dataset": DATASET_VERSION,
    "data_sha256": hashlib.sha256((ROOT / "Data" / f"{DATASET_VERSION}.csv").read_bytes()).hexdigest(),
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "training_pairs": len(train_examples),
    "packages": {
        name: importlib.metadata.version(name)
        for name in ("torch", "transformers", "datasets", "peft", "accelerate")
    },
}
(save_path / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
print(f"Adapter and tokenizer saved to {save_path}")


In [ ]:
import matplotlib.pyplot as plt

loss_logs = [log for log in trainer.state.log_history if "loss" in log]
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot([log["step"] for log in loss_logs], [log["loss"] for log in loss_logs], label="Training Loss")
ax.set(xlabel="Optimizer steps", ylabel="Loss", title=f"Training loss: {RUN_NAME}")
ax.legend()
ax.grid()
fig.savefig(ROOT / f"loss_plot_{RUN_NAME}.png")
plt.show()
plt.close(fig)


In [ ]:
peft_model.config.use_cache = True
for example in test_examples[:5]:
    answer = generate_answer(peft_model, tokenizer, example["soru"], max_new_tokens=64)
    print(f"Question: {example['soru']}")
    print(f"Generated answer: {answer}")
    print(f"Reference answer: {example['çıktı']}")


## Load a saved adapter in a fresh session

This optional cell loads the matching base weights plus the saved LoRA adapter. Run it in a fresh kernel after the configuration cell to avoid retaining the training model in memory.


In [ ]:
if "trainer" in globals():
    raise RuntimeError("Restart the kernel and run the configuration cell before loading the saved adapter")

from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

adapter_path = ROOT / "saved_models" / RUN_NAME
if not (adapter_path / "adapter_config.json").is_file():
    raise FileNotFoundError(f"No saved adapter found at {adapter_path}")
saved_model = AutoPeftModelForCausalLM.from_pretrained(adapter_path, use_safetensors=True)
saved_tokenizer = AutoTokenizer.from_pretrained(adapter_path, local_files_only=True)
print(generate_answer(saved_model, saved_tokenizer, "Python Nedir?", max_new_tokens=64))
